# Geographical analysis of PCAP files

Find autonomous systems and countries where the services were called

In [ ]:
#from scapy.all import rdpcap, IP, IPv6, TCP, UDP
import pandas as pd
import pcap_tools

In [ ]:
pcap_name = "US-Iowa/llama3_agent_traffic"
#pcap_name = "Qwen_agent_traffic"
pcap_name = "genai/gemini-2.5-flash-20251023-090606"
#pcap_name = "agent_traffic"

df_pcap = pcap_tools.pcap_to_dataframe("../pcap/" + pcap_name + ".pcap")
df_pcap = pcap_tools.purify_traffic(df_pcap)
df_pcap = pcap_tools.filter_local_traffic(df_pcap)

In [ ]:
# Group by source_ip and sum lengths to get bytes_sent
sent = df_pcap.groupby("source_ip")["length"].sum().reset_index()
sent.columns = ["ip_addr", "bytes_sent"]

# Group by destination_ip and sum lengths to get bytes_received
received = df_pcap.groupby("destination_ip")["length"].sum().reset_index()
received.columns = ["ip_addr", "bytes_received"]

# Merge sent and received data
df_summary = pd.merge(sent, received, on="ip_addr", how="outer").fillna(0)

# Optional: Convert to integer (in case of NaN→0.0 float)
df_summary["bytes_sent"] = df_summary["bytes_sent"].astype(int)
df_summary["bytes_received"] = df_summary["bytes_received"].astype(int)

In [ ]:
as_list = []
cc_list = []

for ip in df_summary["ip_addr"]:
    asn, cc = pcap_tools.lookup_ip(ip)
    as_list.append(asn)
    cc_list.append(cc)

# Add results to the DataFrame
df_summary["as"] = as_list
df_summary["countryCode"] = cc_list

In [ ]:
df_as_summary = df_summary.groupby(["as", "countryCode"], dropna=False)[["bytes_sent", "bytes_received"]].sum().reset_index()
df_as_summary = df_as_summary.sort_values("bytes_sent", ascending=False)

In [ ]:
df_as_summary.to_csv("../results/" + pcap_name + ".csv")

## Verification

In [ ]:
# Compare the numbers; should be all equal!

traffic_total = sum(df_pcap['length'])
sent2 = sum(df_summary['bytes_sent'])
sent3 = sum(df_as_summary['bytes_sent'])
received2 = sum(df_summary['bytes_received'])
received3 = sum(df_as_summary['bytes_received'])

print(traffic_total, sent2, sent3, received2, received3)


In [ ]:
pcap_tools.ip_cache